# Show the Overturning Streamfunction in the 79NG Fjord

**Caution:**
This notebook uses all CPUs in parallel.
Your computer may slow down during the computation, which should take less than a minute.

Notebook by Markus Reinert (IOW, 2025–2026, https://orcid.org/0000-0002-3761-8029).

In [ ]:
from io import StringIO
from pathlib import Path
from contextlib import redirect_stderr

import numpy as np
import xarray as xr
from cmocean import cm as cmo
from matplotlib import pyplot as plt
from tqdm.auto import tqdm
from joblib import Parallel, delayed

from pyTEF.calc import calc_bulk_values

from tools.visualization import extend_cbar, cm

## Load model results

In [ ]:
folder = Path("output")
!ls -lh $folder/

### Geometry

In [ ]:
geo = xr.open_dataset(folder / "out_geometry_2d.nc").load()

# Compute masks
geo["mask"] = geo.bathymetry.notnull()
geo.mask.attrs = {"long_name": "ocean mask"}
geo["ice_mask"] = geo.glIceD > 0
geo.ice_mask.attrs = {"long_name": "ice mask"}

# Check grid spacing
dlat = 1 / 240
dlon = 3 / 120
assert np.allclose(np.diff(geo.latc), dlat)
assert np.allclose(np.diff(geo.lonc), dlon)

# In lat–lon plots, use the aspect ratio grille_carree to obtain squared grid cells.
# Since dx and dy are approximately equal, this gives an approximate equal-area map.
grille_carree = dlon / dlat

# Show overview
geo.bathymetry.plot(figsize=(12, 7), cmap=cmo.deep)
geo.glIceD.plot.contour(levels=100 * np.arange(8), cmap=cmo.gray, add_colorbar=True)
plt.gca().set(aspect=grille_carree, title="Bathymetry and ice thickness of the 79NG fjord")
plt.grid(ls=":")

geo

### Zonal transports

#### in z-bins

The array `uu` contains the transport in the bin from `zi_z - dz` to `zi_z`.

In [ ]:
uu = xr.open_dataset(folder / "out_zbin.nc").uu_z_mean.squeeze()

# Determine z-bin spacing
dz = (uu.zi_z.max() - uu.zi_z.min()).data / (uu.zi_z.size - 1)
assert np.allclose(dz, np.diff(uu.zi_z)), "the z-bins are not equally spaced."
z_unit = uu.zi_z.units
print("Spacing of z-bins:", dz, z_unit)

# Remove empty levels at depth
empty_levels = (uu == 0).all(["latc", "lonc"])
for i in range(uu.zi_z.size):
    if empty_levels[i]:
        uu = uu.isel(zi_z=slice(1, None))
    else:
        break

# Apply mask
uu = uu.where(geo.mask)

uu

#### in salinity bins

In [ ]:
with xr.open_dataset(folder / "out_sbin_0.02_cavity.nc") as ds:
    uu_s = ds.uu_s_mean.squeeze()
    Su_s = ds.Sfluxu_s_mean.squeeze()

# Determine z-bin spacing
dS = round((uu_s.salt_s.max() - uu_s.salt_s.min()).data / (uu_s.salt_s.size - 1), 10)
assert np.allclose(dS, np.diff(uu_s.salt_s)), "the z-bins are not equally spaced."
s_unit = uu_s.salt_s.units
print("Spacing of salinity bins:", dS, s_unit)

uu_s

#### in temperature bins

In [ ]:
with xr.open_dataset(folder / "TEF_t220.nc") as ds:
    # Divide by 1000 to convert from m³/s to mSv
    Q_T = ds.Q.T / 1e3
    Q_T.attrs = {"long_name": "longitudinal volume transport in temperature coordinates", "units": "mSv"}
    Qt = ds.Qt / 1e3
Q_T

## Select the cavity

In [ ]:
# If the s-binned dataset includes the cavity only, we can use its extent:
cavity = {"lonc": uu_s.lonc, "latc": uu_s.latc}
# Otherwise, use the following extent:
# cavity = {"lonc": slice(-19), "latc": slice(79.24, 80.26)}
geo = geo.sel(**cavity)
uu = uu.sel(**cavity)
# Check that the cavity is surrounded by land to three sides
assert not geo.mask[0].any(), "there are water points in the first row."
assert not geo.mask[-1].any(), "there are water points in the last row."
assert not geo.mask[:, 0].any(), "there are water points in the first column."
geo.mask.plot()

## Compute streamfunctions

### in $z$-coordinates

#### Integrate $u$-transport meridionally to get $q$

In [ ]:
q = (uu * geo.dyu / dz).sum("latc").rename("q")
assert geo.dyu.units == z_unit, "units of z and y are not the same."
q.attrs = {"long_name": "longitudinal volume transport per meter depth", "units": uu.units}

# Move z-axes to bin-centers for correct plotting
q = q.assign_coords(zc=q.zi_z - dz/2).swap_dims(zi_z="zc")
q.zc.attrs = {"long_name": "z at bin center", "units": z_unit}

print(f"q-range: {q.min():.1f} {q.units} to {q.max():.1f} {q.units}")
q.plot(figsize=(10, 6))
q

#### Integrate $q$ vertically to get $Q$

In [ ]:
# Divide by 1000 to convert from m³/s to mSv, given the units are as expected
assert q.units == "m2/s", "unit of q is not as expected."
assert z_unit == "m", "unit of z is not as expected."
# Switch back to interface coordinates, because Q is integrated from the seafloor up to zi_z
Q = (q * dz).cumsum("zc").swap_dims(zc="zi_z").rename("Q") / 1e3
Q.attrs = {"long_name": "longitudinal volume transport", "units": "mSv"}

print(f"Q-range: {Q.min():.1f} {Q.units} to {Q.max():.1f} {Q.units}")
Q.plot(figsize=(10, 6))
Q

### in $S$-coordinates

The streamfunction in salinity coordinates needs to be integrated from high to low salinities, such that zero transport is at the maximum salinity, i.e., at the seafloor.
We do this by inverting the salinity axis before the integration, which is the cumulative sum, and invert the salinity axis back afterwards.

In [ ]:
invert_s = lambda array: array.isel(salt_s=slice(None, None, -1))
# Divide by 1000 to convert from m³/s to mSv
Q_S = invert_s(invert_s(uu_s * geo.dyu).sum("latc").cumsum("salt_s")).rename("Q_S") / 1e3
Qs = invert_s(invert_s(Su_s).sum("latc").cumsum("salt_s")).rename("Qs") / 1e3
Q_S.attrs = {"long_name": "longitudinal volume transport in salinity coordinates", "units": "mSv"}
print(f"Q-range in salinity coordinates: {Q_S.min():.1f} {Q_S.units} to {Q_S.max():.1f} {Q_S.units}")
Q_S.plot(figsize=(10, 6))
Q_S

## Compute bulk values

In [ ]:
# Define a global threshold for the minimum relevant volume transport Q in mSv
Q_thresh = 1.0
print(f"Q threshold: {Q_thresh:.1f} mSv = {Q_thresh / abs(Q_T).max() * 100:.1f} % of max(|Q|) for T-binning")
print(f"Q threshold: {Q_thresh:.1f} mSv = {Q_thresh / abs(Q_S).max() * 100:.1f} % of max(|Q|) for S-binning")

def compute_bulk_values(c, Q, Qc):
    # Suppress the progressbar of pyTEF
    with redirect_stderr(StringIO()):
        # Flip sign of Q for correct labels of in- and outflow
        bulk = calc_bulk_values(c, -Q, -Qc, Q_thresh)
    return bulk

### for temperature

In [ ]:
# Use all CPUs in parallel
parallel = Parallel(n_jobs=-1, return_as="generator")
generate_results = parallel(
    delayed(compute_bulk_values)(Q_T.coords["T"], Q_T.isel(lon=i), Qt.isel(lon=i)) for i in range(Q_T.lon.size)
)

# Create arrays to store the bulk values of main and total in-/outflow (attributes are wrong)
T_in = xr.full_like(Q_T.lon, np.nan)
T_out = xr.full_like(Q_T.lon, np.nan)
T_in_max = xr.full_like(Q_T.lon, np.nan)
T_out_max = xr.full_like(Q_T.lon, np.nan)

for i, bulk in enumerate(tqdm(generate_results, total=Q_T.lon.size)):
    # Handle the inflows if any
    if bulk.m.size > 0:
        T_in[i] = bulk.Qc_in.sum() / bulk.Qin.sum()
        # Make sure the sign is as expected for the maximum
        assert all(bulk.Qin > 0)
        # Select the maximum inflow and compute its bulk temperature (Lorenz et al. 2019, eq. 12)
        m = bulk.Qin.idxmax("m")
        T_in_max[i] = bulk.Qc_in.sel(m=m) / bulk.Qin.sel(m=m)
    # Handle the outflows if any
    if bulk.n.size > 0:
        T_out[i] = bulk.Qc_out.sum() / bulk.Qout.sum()
        assert all(bulk.Qout < 0)  # opposite sign compared to the inflow case above!
        n = bulk.Qout.idxmin("n")
        T_out_max[i] = bulk.Qc_out.sel(n=n) / bulk.Qout.sel(n=n)

In [ ]:
T_in.plot(label="total inflow")
T_in_max.plot(label="main inflow")
T_out.plot(label="total outflow")
T_out_max.plot(label="main outflow")
plt.title("Bulk temperatures")
plt.legend()

### for salinity

In [ ]:
# Use all CPUs in parallel
parallel = Parallel(n_jobs=-1, return_as="generator")
generate_results = parallel(
    delayed(compute_bulk_values)(Q_S.salt_s, Q_S.isel(lonc=i), Qs.isel(lonc=i)) for i in range(Q_S.lonc.size)
)

# Create arrays to store the bulk values of main and total in-/outflow (attributes are wrong)
S_in = xr.full_like(Q_S.lonc, np.nan)
S_out = xr.full_like(Q_S.lonc, np.nan)
S_in_max = xr.full_like(Q_S.lonc, np.nan)
S_out_max = xr.full_like(Q_S.lonc, np.nan)

for i, bulk in enumerate(tqdm(generate_results, total=Q_S.lonc.size)):
    # Handle the inflows if any
    if bulk.m.size > 0:
        S_in[i] = bulk.Qc_in.sum() / bulk.Qin.sum()
        # Make sure the sign is as expected for the maximum
        assert all(bulk.Qin > 0)
        # Select the maximum inflow and compute its bulk salinity (Lorenz et al. 2019, eq. 12)
        m = bulk.Qin.idxmax("m")
        S_in_max[i] = bulk.Qc_in.sel(m=m) / bulk.Qin.sel(m=m)
    # Handle the outflows if any
    if bulk.n.size > 0:
        S_out[i] = bulk.Qc_out.sum() / bulk.Qout.sum()
        assert all(bulk.Qout < 0)  # opposite sign compared to the inflow case above!
        n = bulk.Qout.idxmin("n")
        S_out_max[i] = bulk.Qc_out.sel(n=n) / bulk.Qout.sel(n=n)

In [ ]:
S_in.plot(label="total inflow")
S_in_max.plot(label="main inflow")
S_out.plot(label="total outflow")
S_out_max.plot(label="main outflow")
plt.title("Bulk salinities")
plt.legend()

## Create the figure

In [ ]:
Q_vmin = min(Q_S.min(), Q_T.min())
Q_vmax = max(Q_S.max(), Q_T.max())

Q_pcolor_kwargs = dict(vmin=Q_vmin, vmax=Q_vmax, cmap=cmo.matter_r, add_colorbar=False)
Q_contour_kwargs = dict(levels=[-30, -20, -10], cmap=cmo.gray_r, linewidths=0.75)

In [ ]:
fig, axs = plt.subplot_mosaic("a.\nbc", sharex=True, constrained_layout=True, figsize=(18*cm, 14*cm), dpi=150)
fig.suptitle("Overturning circulation in the 79° North Glacier fjord", weight="bold")

ax = axs["a"]
imq = q.where(q).plot(ax=ax, vmax=400, cmap=cmo.balance, add_colorbar=False)
Q.plot.contour(ax=ax, **Q_contour_kwargs)
ax.set(title="Volume transport in $z$-coordinates", ylabel="depth, $-z$, in m", ylim=(-700, 0))
ax.yaxis.set_major_formatter(lambda x, pos: -int(x) if x else 0)
# Annotate the in- and outflow
arrow = {"arrowstyle": "fancy", "facecolor": "none", "edgecolor": "0.2", "linewidth": 0.75}
ax.annotate("", (-21.8, -550), (-21.2, -550), arrowprops=arrow)
ax.annotate("", (-21.2, -100), (-21.8, -100), arrowprops=arrow)
ax.text(-21.5, -550 - 20, "inflow", size="small", color=arrow["edgecolor"], va="top", ha="center")
ax.text(-21.5, -100 + 20, "outflow", size="small", color=arrow["edgecolor"], va="bottom", ha="center")

ax = axs["b"]
imQ = Q_S.plot(ax=ax, **Q_pcolor_kwargs)
csQ = Q_S.plot.contour(ax=ax, **Q_contour_kwargs)
S_out_max.plot(ax=ax, color="#80d000", ls="--", lw=2)
S_in_max.plot(ax=ax, color="#00ffff", ls="--", lw=2)
ax.set(title="Stream function in $S$-coordinates", ylabel="salinity, $S$, in g kg$^{-1}$", ylim=(33.3, 34.6))
# Annotate the in- and outflow
ax.annotate("", (-21.8, 34.5), (-21.2, 34.5), arrowprops=arrow)
ax.annotate("", (-21.2, 33.6), (-21.8, 33.6), arrowprops=arrow)

ax = axs["c"]
Q_T.plot(ax=ax, **Q_pcolor_kwargs)
Q_T.plot.contour(ax=ax, **Q_contour_kwargs)
T_out_max.plot(ax=ax, color="#80d000", ls="--", lw=2, label="bulk value of the outflow")
T_in_max.plot(ax=ax, color="#00ffff", ls="--", lw=2, label="bulk value of the inflow")
ax.set(title="Stream function in $T$-coordinates", ylabel="temperature, $T$, in °C", ylim=(-1.25, 2))
# Annotate the in- and outflow
ax.annotate("", (-21.8, 1.75), (-21.2, 1.75), arrowprops=arrow)
ax.annotate("", (-21.2, -0.5), (-21.8, -0.5), arrowprops=arrow)

bbox = {"boxstyle": "square", "facecolor": "0.95", "alpha": 2/3, "edgecolor": "none"}
for letter, ax in axs.items():
    ax.text(0.018, 0.977, f"({letter})", transform=ax.transAxes, va="top", clip_on=True, bbox=bbox)
    if letter == "a":
        ax.set_xlabel("")
    else:
        ax.set_xlabel("longitude")
        ax.xaxis.set_major_formatter(lambda x, pos: f"{-x:.0f}°W")
        ax.invert_yaxis()
    # Add ticks without labels to the right side
    ax2 = ax.secondary_yaxis("right")
    ax2.yaxis.set_major_formatter("")

# Make the positions of the subplots final
fig.canvas.draw()

y0 = axs["a"].get_position().ymin
y1 = axs["a"].get_position().ymax
x0 = axs["c"].get_position().xmin
dx = axs["c"].get_position().width
dy = 0.03

fig.legend(bbox_to_anchor=(x0, y0, dx, 0.1), mode="expand", borderaxespad=0, loc="lower left", handlelength=3)
cax = fig.add_axes((x0, y1 - dy, dx, dy))
label = r"transport per depth, $q = \mathrm{d}Q/\mathrm{d}z$, in m$^2$ s$^{-1}$"
fig.colorbar(imq, cax=cax, orientation="horizontal", **extend_cbar(q.min(), q.max(), *imq.get_clim()), label=label)
cax = fig.add_axes((x0, y0 + 0.19, dx, dy))
fig.colorbar(imQ, cax=cax, orientation="horizontal", label="streamfunction, $Q$, in mSv")
for l, c, lw in zip(csQ.levels, csQ.get_edgecolors(), csQ.get_linewidths()):
    cax.axvline(l, color=c, linewidth=lw)

fig.savefig("figures/fig_7_overturning.png", dpi=600)